# PC-QAOA and PenaltyQAOA Comparison

This notebook reads the fixed COP manifest and saved results. It does not generate COPs or run optimization.

Mean plots show pointwise 95% Student-$t$ confidence intervals across COPs:

$$\bar y \pm t_{0.975,n-1}\frac{s}{\sqrt{n}}.$$

Here, $n$ is the number of COPs and $s$ is their sample standard deviation. Optimizer restarts and measurement shots are not independent observations. Paired comparisons use the per-COP difference between PC-QAOA and PenaltyQAOA. Intervals are omitted when fewer than two COPs contribute.

In [ ]:
from pathlib import Path
from collections import Counter
import json, os, shutil, subprocess, warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import display, Image
from scipy.stats import t

candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "code-tuning"]
REPO = next(path for path in candidates if (path / "final_experiment/config.json").exists())
os.environ.setdefault("MPLCONFIGDIR", str(REPO / "final_experiment/.cache/matplotlib"))

MANIFEST_PATH = Path(os.environ.get(
    "PCQAOA_FINAL_MANIFEST", REPO / "final_experiment/artifacts/manifest.json"))
PC_DIR = Path(os.environ.get(
    "PCQAOA_FINAL_PC_RESULTS", REPO / "final_experiment/results/pcqaoa"))
PENALTY_DIR = Path(os.environ.get(
    "PCQAOA_FINAL_PENALTY_RESULTS", REPO / "final_experiment/results/penaltyqaoa"))
if not (PC_DIR / "run.json").exists():
    PC_DIR = REPO / "final_experiment/results/smoke_pcqaoa"
if not (PENALTY_DIR / "run.json").exists():
    PENALTY_DIR = REPO / "final_experiment/results/smoke_penaltyqaoa"
FIGURE_DIR = Path(os.environ.get(
    "PCQAOA_FINAL_FIGURES", REPO / "final_experiment/results/figures"))
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

COLORS = {"PC-QAOA": "#9063cd", "PenaltyQAOA": "#3e8fb0",
          "Disjoint": "#31748f", "Overlapping": "#eb6f92",
          "Teal": "#31748f", "Gold": "#f6c177", "Gray": "#6e6a86"}
MARKERS = {"PC-QAOA": "o", "PenaltyQAOA": "s"}

mpl.rcParams.update({
    "font.family": "serif", "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "cm", "font.size": 11, "axes.titlesize": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.22, "grid.linewidth": 0.5,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "savefig.facecolor": "white", "savefig.dpi": 300, "pdf.fonttype": 42,
    "pgf.texsystem": "pdflatex", "pgf.rcfonts": False, "text.usetex": False,
})

class FigureWriter:
    def __init__(self, directory, latex=True):
        self.directory = Path(directory)
        self.directory.mkdir(parents=True, exist_ok=True)
        self.latex = latex and bool(shutil.which("pdflatex"))
        self.font_status = ("LaTeX Computer Modern (PGF)" if self.latex else
                            "Matplotlib Computer Modern Math / DejaVu Serif")

    def save(self, figure, name):
        pdf = self.directory / f"{name}.pdf"
        png = self.directory / f"{name}.png"
        figure.tight_layout()
        if self.latex:
            try:
                figure.savefig(pdf, backend="pgf", bbox_inches="tight")
                if shutil.which("pdftocairo"):
                    subprocess.run(["pdftocairo", "-png", "-singlefile", "-r", "300",
                                    str(pdf), str(png.with_suffix(""))],
                                   check=True, capture_output=True)
                else:
                    figure.savefig(png, backend="pgf", bbox_inches="tight")
            except Exception as error:
                self.latex = False
                self.font_status = "Matplotlib Computer Modern Math / DejaVu Serif (LaTeX Failed)"
                warnings.warn(f"LaTeX rendering failed; using Matplotlib fonts: {error}")
        if not self.latex:
            figure.savefig(pdf, bbox_inches="tight")
            figure.savefig(png, bbox_inches="tight")
        (self.directory / "font_status.json").write_text(
            json.dumps({"font": self.font_status}, indent=2) + "\n")
        plt.close(figure)
        display(Image(filename=str(png), width=900))
        return png

writer = FigureWriter(FIGURE_DIR, latex=True)
print("Manifest:", MANIFEST_PATH)
print("PC-QAOA Results:", PC_DIR)
print("PenaltyQAOA Results:", PENALTY_DIR)

## Data Loading and Completion

The manifest defines the 500 COPs shared by both methods. Result files are joined by `instance_id`. A paired comparison includes a COP only when both methods have saved the required result.

In [ ]:
def load_json(path, default=None):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else default

def load_results(directory, method):
    rows, depths = [], []
    directory = Path(directory)
    for path in sorted(directory.glob("final_*/result.json")):
        rows.append(load_json(path))
    for path in sorted(directory.glob("final_*/depths.json")):
        for row in load_json(path, []):
            depths.append({"instance_id": path.parent.name, "method": method,
                           **{key: value for key, value in row.items()
                              if key not in ["angles", "history", "restarts",
                                             "counts", "probabilities"]}})
    return pd.DataFrame(rows), pd.DataFrame(depths)

def completion(directory, expected=None):
    statuses = Counter()
    directory = Path(directory)
    selection = load_json(directory / "selection.json")
    if expected is None:
        expected = len(selection["instance_ids"]) if selection else 500
    for path in directory.glob("final_*/status.json"):
        statuses[load_json(path)["status"]] += 1
    completed = len(list(directory.glob("final_*/result.json")))
    statuses["pending"] = max(0, expected - sum(statuses.values()))
    return {"expected": expected, "completed_results": completed,
            "statuses": dict(statuses)}

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Missing {MANIFEST_PATH}. Generate or copy the local manifest before using this notebook.")
manifest = load_json(MANIFEST_PATH)
instances = pd.DataFrame(manifest["instances"])
pc, pc_depths = load_results(PC_DIR, "PC-QAOA")
penalty, penalty_depths = load_results(PENALTY_DIR, "PenaltyQAOA")
for frame, label in [(pc, "PC-QAOA"), (penalty, "PenaltyQAOA")]:
    if not frame.empty:
        frame["method_label"] = label
for frame in [pc_depths, penalty_depths]:
    if not frame.empty:
        frame["method_label"] = frame["method"]
results = pd.concat([pc, penalty], ignore_index=True) if len(pc) or len(penalty) else pd.DataFrame()
depths = pd.concat([pc_depths, penalty_depths], ignore_index=True) if len(pc_depths) or len(penalty_depths) else pd.DataFrame()

print("Manifest COPs:", len(instances))
print("PC-QAOA:", json.dumps(completion(PC_DIR), indent=2))
print("PenaltyQAOA:", json.dumps(completion(PENALTY_DIR), indent=2))
display(instances[["id", "vcg_id", "vcg_family", "support", "n_x", "m", "o",
                   "n_structural", "n_penalty", "delta"]].head())

## Dataset Composition

Each COP contains one trained $L_{\mathrm{fid}}$ VCG constraint and one or two exactly structurally enforceable constraints. The objective coefficients are integers from $-5$ through $5$. PenaltyQAOA uses

$$\delta_{\mathrm{range}}=U_Q-L_Q+1,$$

where $L_Q$ and $U_Q$ are the sums of the negative and positive QUBO coefficients, respectively.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 6.5))
instances["support"].value_counts().sort_index().plot.bar(
    ax=axes[0, 0], color=COLORS["PC-QAOA"])
axes[0, 0].set(xlabel="VCG Support Size", ylabel="COPs", title="VCG Support Distribution")
instances["vcg_family"].value_counts().sort_index().plot.bar(
    ax=axes[0, 1], color=COLORS["PenaltyQAOA"])
axes[0, 1].set(xlabel="VCG Constraint Family", ylabel="COPs", title="VCG Family Distribution")
axes[0, 1].tick_params(axis="x", labelrotation=25)
instances["n_x"].value_counts().sort_index().plot.bar(
    ax=axes[1, 0], color=COLORS["Teal"])
axes[1, 0].set(xlabel="Decision Variables", ylabel="COPs", title="Decision-Variable Distribution")
cell = instances.groupby(["m", "o"]).size().unstack(fill_value=0)
cell.plot.bar(ax=axes[1, 1], color=[COLORS["Gray"], COLORS["Gold"], COLORS["Overlapping"]])
axes[1, 1].set(xlabel="Constraints per COP", ylabel="COPs", title="Constraint and Overlap Distribution")
axes[1, 1].legend(title="Support Overlaps", frameon=False)
writer.save(fig, "dataset_composition")

exact_counts = Counter(category for values in instances["exact_categories"] for category in values)
partitions = instances.groupby(["n_structural", "n_penalty"]).size().rename("COPs").reset_index()
display(pd.Series(exact_counts, name="Occurrences").sort_index().to_frame())
display(partitions.rename(columns={"n_structural": "Structurally Enforced",
                                   "n_penalty": "Penalized"}))

## Optimal-Solution Probability

For the set $\mathcal X^\star$ of all optimal feasible decision strings,

$$P(\mathrm{opt})=\sum_{x\in\mathcal X^\star}P(x).$$

Slack registers are marginalized before this calculation. Sampled probabilities use 10,000 shots in the full experiment. Exact statevector probabilities are retained as diagnostics.

In [ ]:
def mean_ci(frame, group, value):
    stats = frame.groupby(group)[value].agg(["mean", "std", "count"])
    stats["sem"] = stats["std"] / np.sqrt(stats["count"])
    stats["ci95"] = t.ppf(0.975, stats["count"] - 1) * stats["sem"]
    stats.loc[stats["count"] < 2, "ci95"] = np.nan
    return stats.reset_index()

def depth_plot(frame, value, ylabel, title, name):
    fig, ax = plt.subplots(figsize=(6.8, 4.0))
    for method, subset in frame.groupby("method_label", sort=False):
        stats = mean_ci(subset, "depth", value)
        ax.errorbar(stats["depth"], stats["mean"], yerr=stats["ci95"].fillna(0),
                    color=COLORS[method], marker=MARKERS[method], linewidth=1.5,
                    capsize=3, label=method)
    ax.set(xlabel="QAOA Depth", ylabel=ylabel, title=title)
    if len(frame):
        ax.set_xticks(sorted(frame["depth"].unique()))
    ax.legend(frameon=False)
    writer.save(fig, name)

if depths.empty:
    print("No saved depth results are available.")
else:
    depth_plot(depths, "p_optimal", r"Sampled $P(\mathrm{opt})$",
               "Optimal-Solution Probability by Depth", "p_optimal_by_depth")
    depth_plot(depths, "p_feasible", r"Sampled $P(\mathrm{feasible})$",
               "Feasible Probability by Depth", "p_feasible_by_depth")
    display(mean_ci(depths, ["method_label", "depth"], "p_optimal"))

## Paired Final-Depth Comparison

The table and difference plot compare the two methods on the same COPs. A positive difference means PC-QAOA has higher sampled $P(\mathrm{opt})$. Results are separated by decision size and by disjoint or overlapping constraint supports.

In [ ]:
if depths.empty:
    print("No paired depth results are available.")
else:
    final_depth = int(depths["depth"].max())
    final = depths[depths["depth"] == final_depth]
    paired = final.pivot(index="instance_id", columns="method_label", values="p_optimal")
    required = {"PC-QAOA", "PenaltyQAOA"}
    if not required.issubset(paired.columns):
        paired = pd.DataFrame()
    else:
        paired = paired[["PC-QAOA", "PenaltyQAOA"]].dropna()
        paired = paired.join(instances.set_index("id")[["n_x", "regime", "support", "m", "o"]])
    if paired.empty:
        print("No COP has results from both methods at the final available depth.")
    else:
        paired["Difference"] = paired["PC-QAOA"] - paired["PenaltyQAOA"]
        display(paired.describe())
        fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
        axes[0].scatter(paired["PenaltyQAOA"], paired["PC-QAOA"],
                        c=paired["n_x"], cmap="viridis", s=30, alpha=0.8)
        bound = max(paired[["PC-QAOA", "PenaltyQAOA"]].to_numpy().max(), 1e-3)
        axes[0].plot([0, bound], [0, bound], "--", color=COLORS["Gray"], linewidth=1)
        axes[0].set(xlabel=r"PenaltyQAOA $P(\mathrm{opt})$",
                    ylabel=r"PC-QAOA $P(\mathrm{opt})$",
                    title=f"Paired Results at Depth {final_depth}")
        groups = [paired.loc[paired.regime == name, "Difference"]
                  for name in ["disjoint", "overlapping"]]
        means = [group.mean() for group in groups]
        errors = [t.ppf(0.975, len(group)-1) * group.std(ddof=1) / np.sqrt(len(group))
                  if len(group) > 1 else np.nan for group in groups]
        axes[1].bar(["Disjoint", "Overlapping"], means, yerr=np.nan_to_num(errors),
                    color=[COLORS["Disjoint"], COLORS["Overlapping"]], capsize=4)
        axes[1].axhline(0, color=COLORS["Gray"], linewidth=1)
        axes[1].set(ylabel=r"Paired $P(\mathrm{opt})$ Difference",
                    title="PC-QAOA Minus PenaltyQAOA")
        writer.save(fig, "paired_final_depth")

        by_size = paired.groupby("n_x")[["PC-QAOA", "PenaltyQAOA"]].agg(["mean", "count"])
        display(by_size)

## Sampled and Exact Probabilities

Points on the diagonal have equal sampled and exact probabilities. Differences arise from finite measurement shots.

In [ ]:
if depths.empty:
    print("No probability results are available.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
    for method, subset in depths.groupby("method_label", sort=False):
        axes[0].scatter(subset["exact_p_optimal"], subset["p_optimal"],
                        color=COLORS[method], marker=MARKERS[method],
                        s=22, alpha=0.7, label=method)
        axes[1].scatter(subset["exact_p_feasible"], subset["p_feasible"],
                        color=COLORS[method], marker=MARKERS[method],
                        s=22, alpha=0.7, label=method)
    for ax, label in zip(axes, [r"$P(\mathrm{opt})$", r"$P(\mathrm{feasible})$"]):
        ax.plot([0, 1], [0, 1], "--", color=COLORS["Gray"], linewidth=1)
        ax.set(xlabel=f"Exact {label}", ylabel=f"Sampled {label}")
        ax.legend(frameon=False)
    axes[0].set_title("Optimal-Solution Probability")
    axes[1].set_title("Feasible Probability")
    writer.save(fig, "sampled_exact_probabilities")

## Qubits and Runtime

PC-QAOA uses slack qubits only for constraints assigned to its penalty Hamiltonian. PenaltyQAOA encodes every constraint in its penalty Hamiltonian. Runtime plots report recorded optimizer time and the peak resident memory of each worker process.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
qubits = instances[["n_x", "n_total_pcqaoa", "n_total_penaltyqaoa"]].melt(
    id_vars="n_x", var_name="Method", value_name="Total Qubits")
qubits["Method"] = qubits["Method"].map({"n_total_pcqaoa": "PC-QAOA",
                                           "n_total_penaltyqaoa": "PenaltyQAOA"})
for method, subset in qubits.groupby("Method", sort=False):
    stats = mean_ci(subset, "n_x", "Total Qubits")
    axes[0].errorbar(stats["n_x"], stats["mean"], yerr=stats["ci95"].fillna(0),
                     color=COLORS[method], marker=MARKERS[method], capsize=3,
                     linewidth=1.5, label=method)
axes[0].set(xlabel="Decision Variables", ylabel="Total Qubits",
            title="Qubit Counts")
axes[0].set_xticks(sorted(instances["n_x"].unique()))
axes[0].legend(frameon=False)

if results.empty:
    axes[1].text(0.5, 0.5, "No Completed Results", ha="center", va="center",
                 transform=axes[1].transAxes)
else:
    for method, subset in results.groupby("method_label", sort=False):
        axes[1].scatter(subset["n_total"], subset["optimize_seconds"],
                        color=COLORS[method], marker=MARKERS[method], s=28,
                        alpha=0.75, label=method)
    axes[1].legend(frameon=False)
axes[1].set(xlabel="Simulated Qubits", ylabel="Optimization Time (s)",
            title="Optimization Time")
writer.save(fig, "qubits_runtime")

if not results.empty:
    display(results.groupby("method_label")[["compile_seconds", "optimize_seconds",
                                             "process_peak_rss_mb"]].agg(["mean", "std", "count"]))

## Example Optimization History

Each curve is one optimizer restart for one COP, method, and depth. The retained restart has the lowest final penalized expectation at that depth. The vertical axis is the training objective.

In [ ]:
def history_path(directory):
    paths = sorted(Path(directory).glob("final_*/depths.json"))
    return paths[0] if paths else None

paths = [("PC-QAOA", history_path(PC_DIR)),
         ("PenaltyQAOA", history_path(PENALTY_DIR))]
paths = [(method, path) for method, path in paths if path is not None]
if not paths:
    print("No saved optimization histories are available.")
else:
    fig, axes = plt.subplots(1, len(paths), figsize=(6 * len(paths), 4), squeeze=False)
    for ax, (method, path) in zip(axes[0], paths):
        saved = load_json(path)
        depth = saved[-1]
        history = pd.DataFrame(depth["history"])
        retained = min(depth["restarts"], key=lambda row: row["final_loss"])["restart"]
        for restart, frame in history.groupby("restart"):
            ax.plot(frame["step"], frame["loss"],
                    color=COLORS[method] if restart == retained else COLORS["Gray"],
                    alpha=1.0 if restart == retained else 0.55,
                    linewidth=2.2 if restart == retained else 1.0,
                    label=(f"Restart {int(restart)+1} (Retained)" if restart == retained
                           else f"Restart {int(restart)+1}"))
        ax.set(xlabel="Optimizer Step (Before Update)", ylabel="Penalized Expectation",
               title=f"{method}: {path.parent.name}, Depth {depth['depth']}")
        ax.legend(frameon=False, fontsize=8)
    writer.save(fig, "example_optimization_history")

## Reproducibility

`run.json` records the method, configuration, environment, source hashes, VCG database hash, and dataset-manifest hash. `selection.json` records the COP IDs assigned to the run or shard. A resume is rejected when these identifiers do not match the saved run.

In [ ]:
for method, directory in [("PC-QAOA", PC_DIR), ("PenaltyQAOA", PENALTY_DIR)]:
    print(f"\n{method}")
    run = load_json(directory / "run.json")
    selection = load_json(directory / "selection.json")
    if run is None:
        print("No run metadata found.")
    else:
        print("Run Identifier:", run["identifier"])
        print("Manifest Hash:", run["manifest_hash"])
        print("Profile:", run["config"]["profile"])
        print("Selected COPs:", len(selection["instance_ids"]) if selection else "Unknown")
print("\nFigure Fonts:", writer.font_status)
print("Figure Directory:", FIGURE_DIR)